In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
pd.options.display.max_rows = 20
pd.options.display.max_columns = 90

# Loading sql

In [ ]:
import os, shutil

source = "gas_monitoring.db.example"
target = "gas_monitoring.db"

if not os.path.exists(source):
    raise FileNotFoundError(
        f"{source} not found — did you clone the full repo?"
    )
if os.path.exists(target):
    os.remove(target)
shutil.copy2(source, target)
print(f"Copied {source} → {target}")

In [ ]:
con = sqlite3.connect('gas_monitoring.db')

In [ ]:
cursor = con.cursor()

        # Query the sqlite_master table to get table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        # Fetch all results
table_names = [row[0] for row in cursor.fetchall()]


In [ ]:
query = "SELECT * FROM gas_monitoring"  # Replace 'your_table_name' with the actual table name
df = pd.read_sql_query(query, con)

In [ ]:
df.info()

# Initial Analysis

### Temperature

In [ ]:
display(df['Temperature'].describe())

In [ ]:
df['Temperature'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('Temperature Distribution')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(df['Temperature'], bins=50, color='tomato', edgecolor='white')
ax1.set_title('Temperature — Histogram (raw)')
ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Count')

# Box plot
ax2.boxplot(df['Temperature'].dropna())
ax2.set_title('Temperature — Box Plot (raw)')
ax2.set_ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

In [ ]:
anomalies_temp = df[df['Temperature'] > 60]
print(f'Rows with Temperature > 60°C: {len(anomalies_temp)}')
display(anomalies_temp[['Temperature']].head(10))

> **Insights:**
> - The histogram reveals a bimodal-looking distribution with a long right tail — driven by unrealistic high values (up to 307°C).
> - Box plot shows extreme outliers well above the expected indoor range.
> - **Action:** Replace Temperature > 60°C with `NaN`, then impute with the column median.

### Humidity

In [ ]:
display(df['Humidity'].describe())
display(df['Humidity'].isna().sum() / df.shape[0])
neg_humidity = df[df['Humidity'] < 0]
print(f"Found {len(neg_humidity)} rows with negative Humidity")
display(neg_humidity.head(10))

In [ ]:
df['Humidity'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('Humidity Distribution')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(df['Humidity'].dropna(), bins=40, color='cornflowerblue', edgecolor='white')
ax1.set_title('Humidity — Histogram (raw)')
ax1.set_xlabel('Humidity (%)')
ax1.set_ylabel('Count')

ax2.boxplot(df['Humidity'].dropna())
ax2.set_title('Humidity — Box Plot (raw)')
ax2.set_ylabel('Humidity (%)')

plt.tight_layout()
plt.show()

In [ ]:
neg_humidity = df[df['Humidity'] < 0]
over100_humidity = df[df['Humidity'] > 100]
print(f'Rows with Humidity < 0   : {len(neg_humidity)}')
print(f'Rows with Humidity > 100 : {len(over100_humidity)}')

Insights:

19.3% missing values — above the 5% drop threshold, so imputation is the right strategy.
Negative humidity and values > 100% are physically impossible — clear data entry errors.
Action: Replace invalid values (< 0 or > 100) with NaN, then impute with median.

### CO2_InfraredSensor

In [ ]:
display(df[['CO2_InfraredSensor']].describe())

In [ ]:
df['CO2_InfraredSensor'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('CO2 Infrared Sensor Distribution')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(df['CO2_InfraredSensor'], bins=40, color='mediumseagreen', edgecolor='white')
ax1.set_title('CO2 Infrared Sensor — Histogram')
ax1.set_xlabel('ppm')
ax2.boxplot(df['CO2_InfraredSensor'].dropna())
ax2.set_title('CO2 Infrared Sensor — Box Plot')
ax2.set_ylabel('ppm')
plt.tight_layout()
plt.show()

In [ ]:
# IQR outlier detection
Q1 = df['CO2_InfraredSensor'].quantile(0.25)
Q3 = df['CO2_InfraredSensor'].quantile(0.75)
IQR = Q3 - Q1
lower_co2ir = Q1 - 1.5 * IQR
upper_co2ir = Q3 + 1.5 * IQR
outliers_co2ir = df[(df['CO2_InfraredSensor'] < lower_co2ir) | (df['CO2_InfraredSensor'] > upper_co2ir)]
print(f'IQR bounds: [{lower_co2ir:.1f}, {upper_co2ir:.1f}]')
print(f'Outliers: {len(outliers_co2ir)} ({len(outliers_co2ir)/len(df)*100:.1f}%)')

Insights: No missing values. ~8% IQR outliers exist but CO2 readings can legitimately spike during high activity — these will be retained and considered during modelling.

### CO2_ElectroChemicalSensor

In [ ]:
display(df['CO2_ElectroChemicalSensor'].describe())

In [ ]:
df['CO2_ElectroChemicalSensor'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('CO2 ElectroChemical Sensor Distribution')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(df['CO2_ElectroChemicalSensor'], bins=40, color='orchid', edgecolor='white')
ax1.set_title('CO2 ElectroChemical Sensor — Histogram')
ax1.set_xlabel('ppm')
ax2.boxplot(df['CO2_ElectroChemicalSensor'].dropna())
ax2.set_title('CO2 ElectroChemical Sensor — Box Plot')
ax2.set_ylabel('ppm')
plt.tight_layout()
plt.show()

Insights: Very clean column — no missing values, only 5 IQR outliers (0.1%). Distribution is narrow and well-behaved. No cleaning action required.

### CO_GasSensor

In [ ]:
display(df['CO_GasSensor'].value_counts())
display(df['CO_GasSensor'].isna().sum() / df.shape[0])

In [ ]:
df['CO_GasSensor'].value_counts().plot(kind='bar')
plt.title('CO Gas Sensor Distribution')
plt.show()

### Metal Oxide Sensors (Units 1–4)

In [ ]:
mo_cols = [
    'MetalOxideSensor_Unit1',
    'MetalOxideSensor_Unit2',
    'MetalOxideSensor_Unit3',
    'MetalOxideSensor_Unit4'
]
display(df[mo_cols].describe())

In [ ]:
display(df[mo_cols].isna().sum())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, col in zip(axes.ravel(), mo_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[mo_cols])
plt.title('Metal Oxide Sensor Readings Comparison')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[mo_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Metal Oxide Sensor Correlation')
plt.show()

In [ ]:
sns.pairplot(df[mo_cols + ['Temperature', 'Humidity']])
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
df.groupby('Time of Day')[mo_cols].mean().plot(kind='bar')
plt.title('Average Metal Oxide Readings by Time of Day')
plt.legend(loc='best')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
df.groupby('HVAC Operation Mode')[mo_cols].mean().plot(kind='bar')
plt.title('Average Metal Oxide Readings by HVAC Operation Mode')
plt.legend(loc='best')
plt.xticks(rotation=45)
plt.show()

Insights:

Unit2 has around 14% missing values — above 5% threshold, impute with median.
Units 1, 3, 4 are complete (0 missing).
Units 1 & 3 show a wider spread with notable outliers visible in the box plot.
All four sensors operate on similar value ranges (~300–900), suggesting they measure comparable pollutants.

### HVAC Operation Mode

In [ ]:
display(df['HVAC Operation Mode'].value_counts())

In [ ]:
df['HVAC Operation Mode'].value_counts().plot(kind='bar')
plt.title('HVAC Operation Mode Distribution')
plt.xticks(rotation=45)
plt.show()

Insights:

No missing values, but 23 variant spellings exist for 6 logical categories (e.g., cooling_active, COOLING_ACTIVE, Cooling_Active, Cooling_active).
This is a label inconsistency / data entry issue — all variants mean the same category.
Action: Standardise all labels to lowercase underscore format (e.g., cooling_active).

### Ambient Light Level

In [ ]:
display(df['Ambient Light Level'].value_counts())

In [ ]:
df['Ambient Light Level'].value_counts().plot(kind='bar')
plt.title('Ambient Light Level Distribution')
plt.xticks(rotation=45)
plt.show()

Insights:

10.5% missing — above the 5% drop threshold.
Labels are consistent (no casing issues) — only NaN needs handling.
Missing pattern is roughly uniform across Time of Day and HVAC modes (MCAR).
Dominant category is very_bright (33.9%).
Action: Mode imputation with `very_bright` rather than dropping 10% of rows.


### Activity Level

In [ ]:
display(df['Activity Level'].value_counts())

In [ ]:
df['Activity Level'].value_counts().plot(kind='bar')
plt.title('Activity Level Distribution')
plt.xticks(rotation=45)
plt.show()

Insights:

No missing values, but 3 variant spellings exist: ModerateActivity (no space), Low_Activity (underscore), LowActivity (no separator).
Action: Standardise to canonical form: 'Low Activity', 'Moderate Activity', 'High Activity'.

### Time of Day


In [ ]:
display(df['Time of Day'].value_counts())
print(f"Missing: {df['Time of Day'].isna().sum()}")


Insights:

Three consistent categories: morning, afternoon, night.
No missing values, no casing inconsistencies.
No cleaning needed — column is ready for one-hot encoding.


### Session ID


In [ ]:
print(f"Unique IDs: {df['Session ID'].nunique()} / {len(df)} rows")
print(f"Missing: {df['Session ID'].isna().sum()}")


Insights:

Session ID is a unique row identifier (10000 unique / 10000 rows).
No missing values. Has no predictive value — will be dropped during modeling.


# Column by Column Data Cleaning

### Temperature

In [ ]:
# Replace unrealistic Temperature values (> 60°C) with NaN
df.loc[df['Temperature'] > 60, 'Temperature'] = np.nan
temp_median = df['Temperature'].median()
df['Temperature'] = df['Temperature'].fillna(temp_median)

In [ ]:
display(df['Temperature'].describe())

In [ ]:
display(df['Temperature'].isna().sum())

### Humidity

In [ ]:
# Replace physically impossible Humidity values (< 0 or > 100) with NaN
df.loc[df['Humidity'] < 0, 'Humidity'] = np.nan
df.loc[df['Humidity'] > 100, 'Humidity'] = np.nan
humidity_median = df['Humidity'].median()
df['Humidity'] = df['Humidity'].fillna(humidity_median)

In [ ]:
display(df['Humidity'].describe())

In [ ]:
display(df['Humidity'].isna().sum())

### CO2_InfraredSensor

In [ ]:
# Clip negative CO2 values to 0 (physically impossible — sensor calibration error)
df.loc[df['CO2_InfraredSensor'] < 0, 'CO2_InfraredSensor'] = np.nan
ir_median = df['CO2_InfraredSensor'].median()
df['CO2_InfraredSensor'] = df['CO2_InfraredSensor'].fillna(ir_median)
display(pd.DataFrame({'Remaining NaN': [df['CO2_InfraredSensor'].isna().sum()]}))


### CO2_ElectroChemicalSensor

In [ ]:
#there is no error or missing values for CO2 ElectroChemical Sensor
display(df['CO2_ElectroChemicalSensor'].isna().sum())

### CO_GasSensor

In [ ]:
co_mode = df['CO_GasSensor'].mode()[0]
df['CO_GasSensor'] = df['CO_GasSensor'].fillna(co_mode)
df['CO_GasSensor'] = df['CO_GasSensor'].astype(int)

In [ ]:
display(df['CO_GasSensor'].isna().sum())

In [ ]:
display(df['CO_GasSensor'].value_counts())

### Metal Oxide Sensors

In [ ]:
mo_cols = [
    'MetalOxideSensor_Unit1',
    'MetalOxideSensor_Unit2',
    'MetalOxideSensor_Unit3',
    'MetalOxideSensor_Unit4'
]
display(df[mo_cols].isna().sum())

In [ ]:
# Unit2 has ~14% missing values, impute with median
mo_median = df['MetalOxideSensor_Unit2'].median()
df['MetalOxideSensor_Unit2'] = df['MetalOxideSensor_Unit2'].fillna(mo_median)

In [ ]:
display(df[mo_cols].isna().sum())

### HVAC Operation Mode

In [ ]:
display(df['HVAC Operation Mode'].isna().sum())

In [ ]:
# Standardise HVAC Operation Mode labels to lowercase underscore format
hvac_map = {
    'cooling_active': 'cooling_active', 'COOLING_ACTIVE': 'cooling_active', 'Cooling_Active': 'cooling_active', 'Cooling_active': 'cooling_active',
    'heating_active': 'heating_active', 'HEATING_ACTIVE': 'heating_active', 'Heating_Active': 'heating_active', 'Heating_active': 'heating_active',
    'maintenance_mode': 'maintenance_mode', 'MAINTENANCE_MODE': 'maintenance_mode', 'Maintenance_Mode': 'maintenance_mode', 'Maintenance_mode': 'maintenance_mode',
    'eco_mode': 'eco_mode', 'ECO_MODE': 'eco_mode', 'Eco_Mode': 'eco_mode', 'Eco_mode': 'eco_mode', 'Eco': 'eco_mode',
    'off': 'off', 'OFF': 'off', 'Off': 'off',
    'ventilation_only': 'ventilation_only', 'VENTILATION_ONLY': 'ventilation_only', 'Ventilation_Only': 'ventilation_only', 'Ventilation_only': 'ventilation_only'
}
df['HVAC Operation Mode'] = df['HVAC Operation Mode'].map(hvac_map).fillna('other')
print('Standardised HVAC labels:', df['HVAC Operation Mode'].unique())
display(df['HVAC Operation Mode'].value_counts())


### Ambient Light Level

In [ ]:
# --- Diagnose missing pattern ---
light_null = df['Ambient Light Level'].isna()
print('NaN by Time of Day:')
display(df.groupby('Time of Day')['Ambient Light Level'].apply(lambda x: x.isna().sum()))
print('NaN by HVAC Operation Mode:')
display(df.groupby('HVAC Operation Mode')['Ambient Light Level'].apply(lambda x: x.isna().sum()))

# --- Impute with mode (very_bright) to avoid dropping 10% of rows ---
ambient_mode = df['Ambient Light Level'].mode()[0]
print(f"Imputing NaN with mode: {ambient_mode}")
df['Ambient Light Level'] = df['Ambient Light Level'].fillna(ambient_mode)
display(df['Ambient Light Level'].value_counts())


### Activity Level

In [ ]:
display(df['Activity Level'].isna().sum())
display(df['Activity Level'].value_counts())

In [ ]:
# Standardise Activity Level labels to canonical form
activity_map = {
    'Low Activity': 'Low Activity', 'Low_Activity': 'Low Activity', 'LowActivity': 'Low Activity',
    'Moderate Activity': 'Moderate Activity', 'ModerateActivity': 'Moderate Activity',
    'High Activity': 'High Activity'
}
df['Activity Level'] = df['Activity Level'].map(activity_map)
print('Standardised Activity Level labels:', df['Activity Level'].unique())
display(df['Activity Level'].value_counts())


## Cleaned info

In [ ]:
df.info()

## Export clean database

In [ ]:
# Save cleaned database locally

In [ ]:
clean_conn = sqlite3.connect('gas_monitoring_cleaned.db')
df.to_sql('gas_monitoring', clean_conn, if_exists='replace', index=False)
clean_conn.close()
print('Cleaned database saved to gas_monitoring_cleaned.db')
